In [0]:
!pip install openpyxl

In [0]:
import pandas as pd
import glob
import re
from pyspark.sql.functions import col, datediff, when, concat, coalesce, lit, date_format,to_timestamp,udf,to_date
from pyspark.sql.types import IntegerType
from datetime import timedelta, datetime

In [0]:
def extract_date_from_filename(filename):
    """Extract date from filename in format 'dd.mm.yyyy'."""
    match = re.search(r'(\d{2})\.(\d{2})\.(\d{4})\.xlsx', filename)
    if match:
        day, month, year = match.groups()
        return f'{year}-{month}-{day}'  # Format as YYYY-MM-DD
    return None

def add_reporting_date(file_paths):
    # Create an empty list to store DataFrames
    dfs = []
    for file in file_paths:
        # Extract the filename
        filename = file.split('/')[-1]
        
        # Read the Excel file
        df = pd.read_excel(file, dtype=str)
        
        # Extract date from filename for all files
        reporting_date = extract_date_from_filename(filename)
        df['Reporting Date'] = reporting_date
        
        dfs.append(df)
    return dfs
    
# Define UDF to calculate working days
def network_days(start_date, end_date):
    if start_date is None or end_date is None:
        return None

    # Define weekend days (Saturday=5, Sunday=6)
    weekend = {5, 6}
    working_days_count = 0

    # Iterate through the date range
    current_date = start_date
    while current_date <= end_date:
        if current_date.weekday() not in weekend:
            working_days_count += 1
        current_date += timedelta(days=1)

    return working_days_count

In [0]:
# main path
source_path = "/dbfs/mnt/stppeedp/ppeedp/landing/data0/staging/eag/ey/ap_automation/"

# Define the path to write the output 
destination_path = 'dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/me_resolved'
me_resolved_paths = source_path + 'manage_engine_resolved/ME*.xlsx'

In [0]:
# Define the path to your Excel files
file_paths = glob.glob(me_resolved_paths)

# Add Reporting Date Column
dfs = add_reporting_date(file_paths)

# Concatenate all DataFrames
combined_df = pd.concat(dfs, ignore_index=True)

# Convert to Spark DataFrame
me_resolved_df = spark.createDataFrame(combined_df)

# Register UDF
network_days_udf = udf(network_days, IntegerType())

me_resolved_df = me_resolved_df.withColumn(
    "Created Date", 
    when(
        col("Created Time").contains("/"),  # Check if the date contains "/"
        to_date(date_format(to_timestamp(col("Created Time"), "dd/MM/yyyy hh:mm a"), "yyyy-MM-dd"))
    ).otherwise(
        to_date(date_format(to_timestamp(col("Created Time"), "yyyy-MM-dd HH:mm:ss"), "yyyy-MM-dd"))
    )
    ).withColumn(
        "DOC/Non DOC", when(col("Group").contains("DOC"), "DOC").otherwise("NON DOC")
    ).withColumn(
    "Resolved Date",
    when(
        col("Resolved Time").contains("/"),  # Check if the date contains "/"
        to_date(date_format(to_timestamp(col("Resolved Time"), "dd/MM/yyyy hh:mm a"), "yyyy-MM-dd"))
    ).otherwise(
        to_date(date_format(to_timestamp(col("Resolved Time"), "yyyy-MM-dd HH:mm:ss"), "yyyy-MM-dd"))
    )
    ).withColumn(
        "Last_Update_Time",
        when(
        col("Last Update Time").contains("/"),  # Check if the date contains "/"
        to_date(date_format(to_timestamp(col("Last Update Time"), "dd/MM/yyyy hh:mm a"), "yyyy-MM-dd"))
    ).otherwise(
        to_date(date_format(to_timestamp(col("Last Update Time"), "yyyy-MM-dd HH:mm:ss"), "yyyy-MM-dd"))
    )
    ).withColumn(
        "Difference in Days (Inc Weekends)",
        coalesce(datediff(col("Resolved Date"),col("Last_Update_Time")), lit(0))
    ).withColumn("Difference in Days (Excl Weekends)", network_days_udf(col("Last_Update_Time"), col("Resolved Date"))).drop("Last_Update_Time")

# Write the data to the destination
me_resolved_df.write.mode("overwrite").parquet(destination_path)
print(f"Data has been successfully processed and written at {destination_path}")